In [1]:
# ===========================================
# Import Libraries
# Notebook 3: CLASS IMBALANCE
# ===========================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print("✅ Libraries imported successfully")

print("=" * 70)
print("NOTEBOOK 3 — INITIALIZATION")
print("=" * 70)
# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


print(f"Processed data directory:\n{PROCESSED_DIR}")

✅ Libraries imported successfully
NOTEBOOK 3 — INITIALIZATION
Processed data directory:
..\data\processed


In [2]:
# ============================================================
# LOAD NOTEBOOK 2 PROCESSED DATASETS
# ============================================================

train_binary_processed = pd.read_csv(
    PROCESSED_DIR / "train_binary_processed.csv"
)

test_binary_processed = pd.read_csv(
    PROCESSED_DIR / "test_binary_processed.csv"
)

train_multiclass_processed = pd.read_csv(
    PROCESSED_DIR / "train_multiclass_processed.csv"
)

test_multiclass_processed = pd.read_csv(
    PROCESSED_DIR / "test_multiclass_processed.csv"
)

print("=" * 70)
print("NOTEBOOK 2 DATASETS LOADED")
print("=" * 70)

print(
    f"Binary training:     {train_binary_processed.shape}"
)

print(
    f"Binary testing:      {test_binary_processed.shape}"
)

print(
    f"Multiclass training: {train_multiclass_processed.shape}"
)

print(
    f"Multiclass testing:  {test_multiclass_processed.shape}"
)

NOTEBOOK 2 DATASETS LOADED
Binary training:     (468, 62)
Binary testing:      (117, 62)
Multiclass training: (468, 62)
Multiclass testing:  (117, 62)


In [3]:
# ============================================================
#CELL 3-- SEPARATE PREDICTORS AND TARGETS
#       Binary classification: At-Risk vs Not-At-Risk
# ============================================================

BINARY_TARGET = "RISK_BINARY"
MULTICLASS_TARGET = "RISK_LABEL"

X_train_binary = train_binary_processed.drop(
    columns=[BINARY_TARGET]
)

y_train_binary = train_binary_processed[
    BINARY_TARGET
].copy()

X_test_binary = test_binary_processed.drop(
    columns=[BINARY_TARGET]
)

y_test_binary = test_binary_processed[
    BINARY_TARGET
].copy()


# Multiclass
X_train_multiclass = train_multiclass_processed.drop(
    columns=[MULTICLASS_TARGET]
)

y_train_multiclass = train_multiclass_processed[
    MULTICLASS_TARGET
].copy()

X_test_multiclass = test_multiclass_processed.drop(
    columns=[MULTICLASS_TARGET]
)

y_test_multiclass = test_multiclass_processed[
    MULTICLASS_TARGET
].copy()


print("=" * 70)
print("PREDICTOR / TARGET SEPARATION")
print("=" * 70)

print(f"X_train_binary:      {X_train_binary.shape}")
print(f"y_train_binary:      {y_train_binary.shape}")

print(f"X_test_binary:       {X_test_binary.shape}")
print(f"y_test_binary:       {y_test_binary.shape}")

print(f"X_train_multiclass:  {X_train_multiclass.shape}")
print(f"y_train_multiclass:  {y_train_multiclass.shape}")

print(f"X_test_multiclass:   {X_test_multiclass.shape}")
print(f"y_test_multiclass:   {y_test_multiclass.shape}")

PREDICTOR / TARGET SEPARATION
X_train_binary:      (468, 61)
y_train_binary:      (468,)
X_test_binary:       (117, 61)
y_test_binary:       (117,)
X_train_multiclass:  (468, 61)
y_train_multiclass:  (468,)
X_test_multiclass:   (117, 61)
y_test_multiclass:   (117,)


In [4]:
# ============================================================
# CELL 4 -TARGET CODING VALIDATION
# ============================================================

print("=" * 70)
print("TARGET CODING VALIDATION")
print("=" * 70)

print("\nBINARY TARGET")
print("Unique values:", sorted(y_train_binary.unique()))

print("\nBinary training distribution:")
print(
    y_train_binary
    .value_counts()
    .sort_index()
)

print("\nBinary testing distribution:")
print(
    y_test_binary
    .value_counts()
    .sort_index()
)


print("\nMULTICLASS TARGET")
print("Unique values:", sorted(y_train_multiclass.unique()))

print("\nMulticlass training distribution:")
print(
    y_train_multiclass
    .value_counts()
    .sort_index()
)

print("\nMulticlass testing distribution:")
print(
    y_test_multiclass
    .value_counts()
    .sort_index()
)

TARGET CODING VALIDATION

BINARY TARGET
Unique values: [np.int64(0), np.int64(1)]

Binary training distribution:
RISK_BINARY
0    244
1    224
Name: count, dtype: int64

Binary testing distribution:
RISK_BINARY
0    61
1    56
Name: count, dtype: int64

MULTICLASS TARGET
Unique values: ['High Risk', 'Low Risk', 'Moderate Risk']

Multiclass training distribution:
RISK_LABEL
High Risk        135
Low Risk         244
Moderate Risk     89
Name: count, dtype: int64

Multiclass testing distribution:
RISK_LABEL
High Risk        25
Low Risk         61
Moderate Risk    31
Name: count, dtype: int64


In [5]:
# ============================================================
# CELL 5 - CLASS IMBALANCE ASSESSMENT
# ============================================================

def imbalance_report(y, name):

    counts = y.value_counts().sort_index()
    proportions = y.value_counts(
        normalize=True
    ).sort_index()

    majority_count = counts.max()
    minority_count = counts.min()

    imbalance_ratio = (
        majority_count / minority_count
    )

    print("\n" + "-" * 60)
    print(name)
    print("-" * 60)

    for cls in counts.index:
        print(
            f"Class {cls}: "
            f"{counts[cls]} "
            f"({proportions[cls] * 100:.2f}%)"
        )

    print(
        f"Imbalance ratio: "
        f"{imbalance_ratio:.3f}"
    )


imbalance_report(
    y_train_binary,
    "BINARY TRAINING"
)

imbalance_report(
    y_test_binary,
    "BINARY TESTING"
)

imbalance_report(
    y_train_multiclass,
    "MULTICLASS TRAINING"
)

imbalance_report(
    y_test_multiclass,
    "MULTICLASS TESTING"
)


------------------------------------------------------------
BINARY TRAINING
------------------------------------------------------------
Class 0: 244 (52.14%)
Class 1: 224 (47.86%)
Imbalance ratio: 1.089

------------------------------------------------------------
BINARY TESTING
------------------------------------------------------------
Class 0: 61 (52.14%)
Class 1: 56 (47.86%)
Imbalance ratio: 1.089

------------------------------------------------------------
MULTICLASS TRAINING
------------------------------------------------------------
Class High Risk: 135 (28.85%)
Class Low Risk: 244 (52.14%)
Class Moderate Risk: 89 (19.02%)
Imbalance ratio: 2.742

------------------------------------------------------------
MULTICLASS TESTING
------------------------------------------------------------
Class High Risk: 25 (21.37%)
Class Low Risk: 61 (52.14%)
Class Moderate Risk: 31 (26.50%)
Imbalance ratio: 2.440


In [6]:
# ============================================================
# CELL 6 - IMBALANCE STRATEGY DECISION
# ============================================================

print("=" * 70)
print("CLASS-IMBALANCE STRATEGY")
print("=" * 70)

binary_ratio = (
    y_train_binary.value_counts().max()
    / y_train_binary.value_counts().min()
)

multiclass_counts = y_train_multiclass.value_counts()

multiclass_ratio = (
    multiclass_counts.max()
    / multiclass_counts.min()
)

print("\nPRIMARY BINARY TASK")
print(f"Imbalance ratio: {binary_ratio:.3f}")

if binary_ratio < 1.5:
    binary_strategy = "NO_RESAMPLING"
    print("✓ Binary data considered sufficiently balanced")
    print("✓ No SMOTE/SMOTENC will be applied")
else:
    binary_strategy = "IMBALANCE_HANDLING_REQUIRED"
    print("⚠ Binary imbalance requires further consideration")


print("\nSECONDARY MULTICLASS TASK")
print(f"Imbalance ratio: {multiclass_ratio:.3f}")

if multiclass_ratio >= 2:
    multiclass_strategy = "CLASS_WEIGHTING"
    print("✓ Moderate multiclass imbalance detected")
    print("✓ Class-weighted learning will be considered")
else:
    multiclass_strategy = "NO_RESAMPLING"
    print("✓ Multiclass imbalance considered limited")


print("\n" + "=" * 70)
print("DECISION")
print("=" * 70)

print(f"Binary strategy:     {binary_strategy}")
print(f"Multiclass strategy: {multiclass_strategy}")

CLASS-IMBALANCE STRATEGY

PRIMARY BINARY TASK
Imbalance ratio: 1.089
✓ Binary data considered sufficiently balanced
✓ No SMOTE/SMOTENC will be applied

SECONDARY MULTICLASS TASK
Imbalance ratio: 2.742
✓ Moderate multiclass imbalance detected
✓ Class-weighted learning will be considered

DECISION
Binary strategy:     NO_RESAMPLING
Multiclass strategy: CLASS_WEIGHTING


In [7]:
# ============================================================
# CELL 7- CREATE FINAL NOTEBOOK 3 MODELLING DATASETS
# ============================================================

# ------------------------------------------------------------
# PRIMARY BINARY TASK
# ------------------------------------------------------------

X_binary_model = X_train_binary.copy()
y_binary_model = y_train_binary.copy()

X_binary_test = X_test_binary.copy()
y_binary_test = y_test_binary.copy()


# ------------------------------------------------------------
# SECONDARY MULTICLASS TASK
# ------------------------------------------------------------

X_multiclass_model = X_train_multiclass.copy()
y_multiclass_model = y_train_multiclass.copy()

X_multiclass_test = X_test_multiclass.copy()
y_multiclass_test = y_test_multiclass.copy()


print("=" * 70)
print("FINAL MODELLING DATASETS")
print("=" * 70)

print("\nBINARY")
print(f"Training predictors: {X_binary_model.shape}")
print(f"Training target:     {y_binary_model.shape}")
print(f"Testing predictors:  {X_binary_test.shape}")
print(f"Testing target:      {y_binary_test.shape}")

print("\nMULTICLASS")
print(f"Training predictors: {X_multiclass_model.shape}")
print(f"Training target:     {y_multiclass_model.shape}")
print(f"Testing predictors:  {X_multiclass_test.shape}")
print(f"Testing target:      {y_multiclass_test.shape}")

FINAL MODELLING DATASETS

BINARY
Training predictors: (468, 61)
Training target:     (468,)
Testing predictors:  (117, 61)
Testing target:      (117,)

MULTICLASS
Training predictors: (468, 61)
Training target:     (468,)
Testing predictors:  (117, 61)
Testing target:      (117,)


In [8]:
# ============================================================
# CELL -8 TEST-SET INTEGRITY CHECK
# ============================================================

print("=" * 70)
print("TEST-SET INTEGRITY CHECK")
print("=" * 70)

# Confirm no resampling changed test observations
assert len(X_binary_test) == 117
assert len(y_binary_test) == 117

assert len(X_multiclass_test) == 117
assert len(y_multiclass_test) == 117

# Confirm no missing values
assert X_binary_test.isnull().sum().sum() == 0
assert X_multiclass_test.isnull().sum().sum() == 0

print("✓ Binary test set: 117 observations")
print("✓ Multiclass test set: 117 observations")
print("✓ No test-set resampling")
print("✓ No test-set missing values")
print("\n✓ TEST SET REMAINS UNTOUCHED")

TEST-SET INTEGRITY CHECK
✓ Binary test set: 117 observations
✓ Multiclass test set: 117 observations
✓ No test-set resampling
✓ No test-set missing values

✓ TEST SET REMAINS UNTOUCHED


In [9]:
# ============================================================
#CALCULATE MULTICLASS CLASS WEIGHTS
# CELL 9- MULTICLASS CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

classes = np.sort(
    y_multiclass_model.unique()
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_multiclass_model
)

multiclass_class_weights = dict(
    zip(classes, weights)
)

print("=" * 70)
print("MULTICLASS CLASS WEIGHTS")
print("=" * 70)

for cls, weight in multiclass_class_weights.items():
    print(f"Class {cls}: {weight:.4f}")

MULTICLASS CLASS WEIGHTS
Class High Risk: 1.1556
Class Low Risk: 0.6393
Class Moderate Risk: 1.7528


In [10]:
# ============================================================
# CELL 10 -- NOTEBOOK 3 — FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 3 — FINAL VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Shape validation
# ------------------------------------------------------------

assert X_binary_model.shape == (468, 61)
assert y_binary_model.shape == (468,)

assert X_binary_test.shape == (117, 61)
assert y_binary_test.shape == (117,)

assert X_multiclass_model.shape == (468, 61)
assert y_multiclass_model.shape == (468,)

assert X_multiclass_test.shape == (117, 61)
assert y_multiclass_test.shape == (117,)

print("✓ Dataset dimensions validated")


# ------------------------------------------------------------
# 2. Missingness validation
# ------------------------------------------------------------

assert X_binary_model.isnull().sum().sum() == 0
assert X_binary_test.isnull().sum().sum() == 0

assert X_multiclass_model.isnull().sum().sum() == 0
assert X_multiclass_test.isnull().sum().sum() == 0

print("✓ No missing predictor values")


# ------------------------------------------------------------
# 3. Numeric predictor validation
# ------------------------------------------------------------

assert X_binary_model.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert X_multiclass_model.select_dtypes(
    exclude=np.number
).shape[1] == 0

print("✓ All predictors numeric")


# ------------------------------------------------------------
# 4. Train/test feature alignment
# ------------------------------------------------------------

assert list(X_binary_model.columns) == list(
    X_binary_test.columns
)

assert list(X_multiclass_model.columns) == list(
    X_multiclass_test.columns
)

print("✓ Train/test feature alignment confirmed")


# ------------------------------------------------------------
# 5. Binary imbalance strategy
# ------------------------------------------------------------

assert binary_strategy == "NO_RESAMPLING"

print("✓ Binary strategy locked: NO_RESAMPLING")


# ------------------------------------------------------------
# 6. Multiclass strategy
# ------------------------------------------------------------

assert multiclass_strategy == "CLASS_WEIGHTING"

print("✓ Multiclass strategy locked: CLASS_WEIGHTING")


# ------------------------------------------------------------
# 7. Test-set size preservation
# ------------------------------------------------------------

assert len(X_binary_test) == 117
assert len(X_multiclass_test) == 117

print("✓ Test sets remain at 117 observations")


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✓ NOTEBOOK 3 FINAL VALIDATION PASSED")
print("=" * 70)

print("\nPRIMARY BINARY TASK")
print("  Training:       468 × 61")
print("  Testing:        117 × 61")
print("  Strategy:       NO_RESAMPLING")

print("\nSECONDARY MULTICLASS TASK")
print("  Training:       468 × 61")
print("  Testing:        117 × 61")
print("  Strategy:       CLASS_WEIGHTING")

print("\n✓ TEST DATA REMAINS UNTOUCHED")
print("✓ NO SYNTHETIC DATA GENERATED")
print("✓ NOTEBOOK 3 READY FOR FINAL SAVE")

NOTEBOOK 3 — FINAL VALIDATION
✓ Dataset dimensions validated
✓ No missing predictor values
✓ All predictors numeric
✓ Train/test feature alignment confirmed
✓ Binary strategy locked: NO_RESAMPLING
✓ Multiclass strategy locked: CLASS_WEIGHTING
✓ Test sets remain at 117 observations

✓ NOTEBOOK 3 FINAL VALIDATION PASSED

PRIMARY BINARY TASK
  Training:       468 × 61
  Testing:        117 × 61
  Strategy:       NO_RESAMPLING

SECONDARY MULTICLASS TASK
  Training:       468 × 61
  Testing:        117 × 61
  Strategy:       CLASS_WEIGHTING

✓ TEST DATA REMAINS UNTOUCHED
✓ NO SYNTHETIC DATA GENERATED
✓ NOTEBOOK 3 READY FOR FINAL SAVE


In [12]:
# ============================================================
# CELL 11 - NOTEBOOK 3 — SAVE FINAL MODELLING DATASETS
# ============================================================

X_binary_model_with_target = X_binary_model.copy()
X_binary_model_with_target["RISK_BINARY"] = y_binary_model.values

X_binary_test_with_target = X_binary_test.copy()
X_binary_test_with_target["RISK_BINARY"] = y_binary_test.values

X_binary_model_with_target.to_csv(
    PROCESSED_DIR / "train_binary_model.csv",
    index=False
)

X_binary_test_with_target.to_csv(
    PROCESSED_DIR / "test_binary_model.csv",
    index=False
)


# ------------------------------------------------------------
# Save multiclass datasets
# ------------------------------------------------------------

X_multiclass_model_with_target = X_multiclass_model.copy()
X_multiclass_model_with_target["RISK_LABEL"] = (
    y_multiclass_model.values
)

X_multiclass_test_with_target = X_multiclass_test.copy()
X_multiclass_test_with_target["RISK_LABEL"] = (
    y_multiclass_test.values
)

X_multiclass_model_with_target.to_csv(
    PROCESSED_DIR / "train_multiclass_model.csv",
    index=False
)

X_multiclass_test_with_target.to_csv(
    PROCESSED_DIR / "test_multiclass_model.csv",
    index=False
)


# ------------------------------------------------------------
# Save multiclass class weights
# ------------------------------------------------------------

weights_df = pd.DataFrame(
    {
        "Class": list(multiclass_class_weights.keys()),
        "Weight": list(multiclass_class_weights.values())
    }
)

weights_df.to_csv(
    PROCESSED_DIR / "multiclass_class_weights.csv",
    index=False
)


# ------------------------------------------------------------
# Confirm files
# ------------------------------------------------------------

print("=" * 70)
print("NOTEBOOK 3 OUTPUTS SAVED")
print("=" * 70)

print("\nBinary:")
print("  ✓ train_binary_model.csv")
print("  ✓ test_binary_model.csv")

print("\nMulticlass:")
print("  ✓ train_multiclass_model.csv")
print("  ✓ test_multiclass_model.csv")

print("\nClass weights:")
print("  ✓ multiclass_class_weights.csv")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 3 COMPLETE")
print("=" * 70)

NOTEBOOK 3 OUTPUTS SAVED

Binary:
  ✓ train_binary_model.csv
  ✓ test_binary_model.csv

Multiclass:
  ✓ train_multiclass_model.csv
  ✓ test_multiclass_model.csv

Class weights:
  ✓ multiclass_class_weights.csv

✓ NOTEBOOK 3 COMPLETE
